<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_32_rest_api_design/note_lesson_32_rest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌦️ Урок 32 — REST: дизайн API на метеоданих

| Крок | Що робимо |
|---|---|
| 0 | запускаємо Meteo API v2 |
| 1 | v1 → v2: знаходимо порушення REST (вправа 1) |
| 2 | статус-коди: 201, 409, 422, 404 (вправа 2) |
| 3 | пагінація (вправа 3) |
| 4 | `PATCH` і `DELETE` (вправи 4–5) |
| 5 | вибір полів (вправа 6) |
| 6 | GraphQL — інший тип API (вправа 7) |
| 7 | клієнт до API (вправа 8) |
| 8 | знайди помилку: 200 з `{"error": …}` (вправа 9) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія, всі типи API (RPC, gRPC, SOAP, WebSocket, SSE, webhook) і схеми — у книзі: [Урок 32](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_32/).

---
## 0. Meteo API v2

Проєкт уроку — пакет `meteo_api` (FastAPI) і клієнт `weather_map/client.py` поруч із цим ноутбуком. У Colab ноутбук відкривається **без** файлів репозиторію, тому клітинка нижче завантажує лише папку уроку з GitHub (`git clone --sparse`) і ставить бібліотеки, яких бракує.

In [ ]:
import importlib.util
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_32_rest_api_design"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("meteo_api"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

packages = {"fastapi": "fastapi", "uvicorn": "uvicorn", "pymetdecoder": "pymetdecoder",
            "strawberry": "strawberry-graphql", "httpx": "httpx"}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "weather_map"))
print("папка уроку:", os.getcwd())

In [ ]:
import requests

from meteo_api.app import start_server

BASE = start_server()
print(BASE)
print(requests.get(f"{BASE}/api/v1/stations", params={"limit": 3}, timeout=5).json())

Щоб вправи не залежали від дня, коли зроблено знімок даних, додамо **умовну станцію** `99001` «Навчальна» не через API, а прямо в сховище, і вісім **умовних** спостережень — уже через API (значення вигадані для вправ).

In [ ]:
from meteo_api.app import app

repo = app.state.repo
repo.add_station("99001", "Навчальна")
OBS = f"{BASE}/api/v1/stations/99001/observations"
for hour, temperature in zip(range(0, 24, 3), [11.2, 10.4, 12.9, 17.5, 20.1, 19.3, 15.8, 13.0]):
    response = requests.post(OBS, json={"time": f"2026-09-20T{hour:02d}:00Z", "temperature": temperature,
                                        "pressure": 1005.0 + hour / 3}, timeout=5)
    print(response.status_code, response.headers["Location"])

---
## 1. v1 → v2

Адреси першої версії API (з `meteo_api/legacy/main.py`):

| v1 | що робить |
|---|---|
| `POST /filter_telegrams/` | пошук спостережень за станцією, датою, годиною |
| `POST /download_telegrams` | завантажити телеграми з ogimet (хвилини) |
| `GET /telegram/ua/3450420249218` | одна телеграма (станція 34504, 2024-09-02, 18 год) |
| `DELETE /telegram/ua/3450420249218` | видалити її |

### Вправа 1

Заповни словник `V2`: для кожного запиту v1 — метод і шлях v2 (див. `GET /openapi.json` або книгу). Формат значення: `"МЕТОД /шлях"`, ідентифікатори — як у прикладі.

In [ ]:
V2 = {
    # YOUR CODE HERE
    # BEGIN SOLUTION
    "POST /filter_telegrams/": "GET /api/v1/stations/34504/observations?hour=18",
    "POST /download_telegrams": "POST /api/v1/imports",
    "GET /telegram/ua/3450420249218": "GET /api/v1/stations/34504/observations/2024-09-02T18:00Z",
    "DELETE /telegram/ua/3450420249218": "DELETE /api/v1/stations/34504/observations/2024-09-02T18:00Z",
    # END SOLUTION
}

assert V2["POST /filter_telegrams/"].startswith("GET /api/v1/stations/34504/observations"), "пошук — це читання"
assert V2["POST /download_telegrams"] == "POST /api/v1/imports"
assert V2["GET /telegram/ua/3450420249218"] == "GET /api/v1/stations/34504/observations/2024-09-02T18:00Z"
assert V2["DELETE /telegram/ua/3450420249218"].startswith("DELETE /api/v1/stations/34504/observations/")
print("✅ Вправа 1 пройдена")

**🔮 Прогноз:** Що має повернути `POST /api/v1/imports`, якщо імпорт триває хвилини?

<details>
<summary>Відповідь</summary>

`202 Accepted` і заголовок `Location` з адресою задачі; клієнт потім перевіряє її стан через `GET`.

</details>

---
## 2. Статус-коди

### Вправа 2

Зроби чотири запити до `OBS` (станція 99001) і збери їхні коди в список `codes` у такому порядку:

1. створити спостереження на `2026-09-21T00:00Z` з температурою 9.5;
2. те саме вдруге;
3. спостереження на `2026-09-21T03:00Z` з температурою 75;
4. `GET` станції `00000`.

In [ ]:
codes = []
# YOUR CODE HERE
# BEGIN SOLUTION
body = {"time": "2026-09-21T00:00Z", "temperature": 9.5}
codes.append(requests.post(OBS, json=body, timeout=5).status_code)
codes.append(requests.post(OBS, json=body, timeout=5).status_code)
codes.append(requests.post(OBS, json={"time": "2026-09-21T03:00Z", "temperature": 75}, timeout=5).status_code)
codes.append(requests.get(f"{BASE}/api/v1/stations/00000", timeout=5).status_code)
# END SOLUTION

assert codes == [201, 409, 422, 404], codes
print("✅ Вправа 2 пройдена")

---
## 3. Пагінація

Відповідь колекції — конверт `{"items", "total", "limit", "offset", "next"}`. `next` — готове посилання на наступну сторінку або `None`.

### Вправа 3

`fetch_all(url, limit)` проходить усі сторінки за посиланнями `next` і повертає пару: `(усі елементи, скільки запитів зроблено)`.

In [ ]:
def fetch_all(url, limit):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    items, requests_made, params = [], 0, {"limit": limit}
    while url:
        page = requests.get(url, params=params, timeout=5).json()
        requests_made += 1
        items += page["items"]
        url, params = page["next"], None
    return items, requests_made
    # END SOLUTION


items, made = fetch_all(OBS, limit=3)
assert len(items) == 9 and made == 3, (len(items), made)
assert [item["time"] for item in items] == sorted(item["time"] for item in items)
items, made = fetch_all(OBS, limit=100)
assert len(items) == 9 and made == 1
print("✅ Вправа 3 пройдена")

**🔮 Прогноз:** Скільки запитів зробить `fetch_all(OBS, limit=4)`?

<details>
<summary>Відповідь</summary>

Три: 4 + 4 + 1 елемент. На третій сторінці `next` стане `None`.

</details>

---
## 4. PATCH і DELETE

### Вправа 4

Виправ температуру спостереження `2026-09-20T12:00Z` на `20.4` запитом `PATCH`. Поверни JSON відповіді в `fixed`.

In [ ]:
url = f"{OBS}/2026-09-20T12:00Z"
# YOUR CODE HERE
# BEGIN SOLUTION
fixed = requests.patch(url, json={"temperature": 20.4}, timeout=5).json()
# END SOLUTION

assert fixed["temperature"] == 20.4
assert fixed["pressure"] == 1009.0, "PATCH не має чіпати інші поля"
print("✅ Вправа 4 пройдена")

### Вправа 5

Видали спостереження `2026-09-21T00:00Z` **двічі** і збери коди відповідей у `delete_codes`; потім перевір `GET` цього спостереження — код у `after`.

In [ ]:
url = f"{OBS}/2026-09-21T00:00Z"
# YOUR CODE HERE
# BEGIN SOLUTION
delete_codes = [requests.delete(url, timeout=5).status_code for _ in range(2)]
after = requests.get(url, timeout=5).status_code
# END SOLUTION

assert delete_codes == [204, 404]
assert after == 404
print("✅ Вправа 5 пройдена")

**🔮 Прогноз:** Коди двох `DELETE` різні. Чому `DELETE` все одно називають ідемпотентним?

<details>
<summary>Відповідь</summary>

Ідемпотентність — про **стан сервера**, а не про код відповіді: після одного чи двох запитів спостереження однаково немає.

</details>

---
## 5. Вибір полів

### Вправа 6

Отримай спостереження станції 99001 лише з полями `time` і `temperature` (параметр `fields`) і поверни їх списком `rows`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
rows = requests.get(OBS, params={"fields": "time,temperature", "limit": 100}, timeout=5).json()["items"]
# END SOLUTION

assert len(rows) == 8
assert all(set(row) == {"time", "temperature"} for row in rows)
assert rows[4] == {"time": "2026-09-20T12:00Z", "temperature": 20.4}
print("✅ Вправа 6 пройдена")

---
## 6. GraphQL — інший тип API

Той самий сервер має `/graphql`: клієнт сам описує поля. Приклад запиту: `{ station(wmo: "34504") { name } }`.

### Вправа 7

Напиши запит `query`, який повертає назву станції `99001` і **два останні** спостереження з полями `time` і `pressure`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
query = '{ station(wmo: "99001") { name observations(last: 2) { time pressure } } }'
# END SOLUTION

data = requests.post(f"{BASE}/graphql", json={"query": query}, timeout=5).json()["data"]
assert data["station"]["name"] == "Навчальна"
assert data["station"]["observations"] == [{"time": "2026-09-20T18:00Z", "pressure": 1011.0},
                                           {"time": "2026-09-20T21:00Z", "pressure": 1012.0}]
print(data)
print("✅ Вправа 7 пройдена")

---
## 7. Клієнт до API

`MeteoClient` (файл `weather_map/client.py`) сам гортає сторінки й перетворює 404 на `StationNotFound` — як `SmachnoClient` в уроці 31.

### Вправа 8

`mean_temperature(client, wmo)` повертає середню температуру станції (округлену до 0.1) або `None`, якщо станції немає.

In [ ]:
from client import MeteoClient, StationNotFound


def mean_temperature(client, wmo):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    try:
        items = client.observations(wmo, fields="temperature")
    except StationNotFound:
        return None
    return round(sum(item["temperature"] for item in items) / len(items), 1)
    # END SOLUTION


client = MeteoClient(BASE)
assert mean_temperature(client, "99001") == 15.1, mean_temperature(client, "99001")
assert mean_temperature(client, "00000") is None
print("✅ Вправа 8 пройдена")

---
## 8. Знайди помилку

```python
@buggy.post("/get_mean_temperature")
def get_mean_temperature(wmo: str):
    try:
        items = repo.list_observations(wmo)
        return {"mean": sum(o["temperature"] for o in items) / len(items)}
    except Exception as e:
        return {"error": str(e)}
```

Для неіснуючої станції цей ендпоінт відповідає `200 {"error": …}`.

### Вправа 9

Напиши правильну версію: `GET /api/v1/stations/{wmo}/temperature/mean` у новому застосунку `fixed_app`. Неіснуюча станція → `404`; станція без спостережень → `404` з поясненням; інакше `200 {"station": wmo, "mean": …}`.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

from meteo_api.storage import NotFoundError

fixed_app = FastAPI()

# YOUR CODE HERE
# BEGIN SOLUTION
@fixed_app.get("/api/v1/stations/{wmo}/temperature/mean")
def mean_temperature_endpoint(wmo: str):
    try:
        items = repo.list_observations(wmo)
    except NotFoundError as error:
        raise HTTPException(404, str(error))
    if not items:
        raise HTTPException(404, f"у станції {wmo} немає спостережень")
    return {"station": wmo, "mean": round(sum(o["temperature"] for o in items) / len(items), 1)}
# END SOLUTION

test = TestClient(fixed_app)
ok = test.get("/api/v1/stations/99001/temperature/mean")
assert ok.status_code == 200 and ok.json() == {"station": "99001", "mean": 15.1}, ok.json()
assert test.get("/api/v1/stations/00000/temperature/mean").status_code == 404
repo.add_station("99002", "Порожня")
assert test.get("/api/v1/stations/99002/temperature/mean").status_code == 404
print("✅ Вправа 9 пройдена")

---
## Самоперевірка

1. Чому пошук — це `GET` з параметрами, а не `POST /filter…`?
2. Які коди означають: створено; вже є; дані неправильні; не знайдено; прийнято, виконується?
3. Навіщо в конверті сторінки `next`?
4. Чим `PATCH` відрізняється від `PUT`?
5. Чим GraphQL відрізняється від REST?

<details>
<summary>Відповіді</summary>

1. Пошук лише читає: `GET` ідемпотентний, кешується, його можна відкрити в браузері; дієслово в URL суперечить ідеї ресурсів.
2. `201` (+ `Location`), `409`, `422`, `404`, `202` (+ `Location`).
3. Клієнт іде за готовим посиланням і не рахує `offset` сам; сервер може змінити спосіб пагінації.
4. `PATCH` змінює лише передані поля, `PUT` замінює ресурс цілком.
5. Один URL, клієнт сам обирає поля й пов'язані дані одним запитом; обов'язкова схема.

</details>

## Далі

- Книга: [Урок 32](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_32/) — усі типи API з прикладами (HTTP+CSV, JSON-RPC, GraphQL, SOAP, gRPC, WebSocket, SSE, webhook), `202 Accepted`, OpenAPI, архітектура й карта погоди на Streamlit.
- Карта: `uvicorn meteo_api.app:app --port 8032` і `streamlit run weather_map/app.py` у папці уроку.
- **Урок 33** — Django intro: MVT, ORM, admin.